In [2]:
import pandas as pd
import numpy as np
from natasha import NamesExtractor, MorphVocab
import pymorphy3
import re

In [ ]:
df = pd.read_excel('export.xlsx')
df

/Users/sergej/PyCharmMiscProject/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Реестр иностранных агентов,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,2025-09-05 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,№ п/п,Полное наименование (прежнее наименование (в с...,Основания для включения,Дата принятия Минюстом России решения о включе...,Дата принятия Минюстом России решения об исклю...,Доменное имя информационного ресурса (при нали...,Тип иностранного агента,Регистрационный номер,ИНН,ОГРН,...,Дата рождения,Полное наименование или ФИО участников,Адрес (место нахождения),Дата опубликования принятого Минюстом России р...,Номер специального счета,Наименование и местонахождение уполномоченного...,Банковский идентификационный код уполномоченно...,Номер корреспондентского счета (субсчета) упол...,Дата открытия специального счета,Дата заключения договора банковского счёта
2,1068,Сетевой проект «Om TV»,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://www.youtube.com/@omtvreal; https://t.m...,Иные объединения лиц,NaN,NaN,NaN,...,NaN,[Омельчук Сергей Сергеевич],NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
3,1067,«Компромат 1»,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://kompromat1.online/; https://kompromat1...,Иные объединения лиц,NaN,NaN,NaN,...,NaN,"[Черненко Константин Евгеньевич, Преснаков Але...",NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
4,1066,Степанова Анна Васильевна,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://t.me/markizaakarabasa; https://www.you...,Физические лица,NaN,292006666153,NaN,...,28.10.1979,NaN,NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1065,5,Региональная общественная организация в защиту...,Статья 32 Федерального закона от 12.01.1996 № ...,05.06.2014,20.02.2017,NaN,Юридические лица,NaN,7709439312,1067799008860,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1066,4,Автономная некоммерческая научно-исследователь...,Статья 32 Федерального закона от 12.01.1996 № ...,05.06.2014,22.05.2015,NaN,Юридические лица,NaN,6452083317,1036405207640,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1067,3,Региональная общественная правозащитная органи...,Статья 32 Федерального закона от 12.01.1996 № ...,05.06.2014,29.02.2016,NaN,Юридические лица,NaN,6150025245,1026100023872,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1068,2,Ассоциация некоммерческих организаций «В защит...,Статья 32 Федерального закона от 12.01.1996 № ...,05.06.2014,13.03.2020,NaN,Юридические лица,NaN,7702295527,1037739618872,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
#оставляем только имена - удаляем организации. Принцип - "Тип иностранного агента" - физ лицо

df = df[df['Unnamed: 6'] == "Физические лица"]
df

,Реестр иностранных агентов,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
4,1066,Степанова Анна Васильевна,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://t.me/markizaakarabasa; https://www.you...,Физические лица,NaN,292006666153,NaN,...,28.10.1979,NaN,NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
5,1065,Смолин Владимир Александрович,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://t.me/smolin_info; https://www.facebook...,Физические лица,NaN,240701390600,NaN,...,14.10.1981,NaN,NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
6,1064,Ицхоки Олег Евгеньевич,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://itskhoki.com; https://www.instagram.co...,Физические лица,NaN,771402711994,NaN,...,07.01.1983,NaN,NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
7,1063,Говорун Сергей Николаевич,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://t.me/cyrilhovorun; https://www.faceboo...,Физические лица,NaN,NaN,NaN,...,28.01.1974,NaN,NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
8,1062,"Несмиян Анатолий Евгеньевич ""El Murid""",Статья 7 Федерального закона от 14.07.2022 № 2...,29.08.2025,NaN,"https://t.me/anatoly_nesmiyan, ID: -1001540992...",Физические лица,NaN,165002994679,NaN,...,08.11.1965,NaN,NaN,29.08.2025,40817810255193220413,Головное отделение Северо-Западного банка №905...,044030653,30101810500000000653,03.09.2025,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
854,216,Савицкая Людмила Алексеевна,Статья 6 Закона Российской Федерации от 27.12....,28.12.2020,NaN,https://vk.com/id25797959; https://lsavitskaya...,Физические лица,NaN,602722507030,NaN,...,22.01.1991,NaN,NaN,NaN,40817810751861520329,Псковское отделение №8630 ПАО Сбербанк,045805602,30101810300000000602,01.03.2025,NaN
855,215,Пономарев Лев Александрович,Статья 6 Закона Российской Федерации от 27.12....,28.12.2020,NaN,https://www.youtube.com/@levzaprava; https://t...,Физические лица,NaN,773404771583,NaN,...,02.09.1941,NaN,NaN,NaN,40817810638700402181,Московский банк ПАО Сбербанк,044525225,30101810400000000225,01.03.2025,NaN
856,214,Маркелов Сергей Евгеньевич,Статья 6 Закона Российской Федерации от 27.12....,28.12.2020,NaN,http://xamin.tilda.ws/cv?fbclid=IwAR0VahYgErIP...,Физические лица,NaN,100122365435,NaN,...,11.10.1986,NaN,NaN,NaN,40817810225861797882,Карельское отделение №8628 ПАО Сбербанк,048602673,30101810600000000673,01.03.2025,NaN
857,213,Камалягин Денис Николаевич,Статья 6 Закона Российской Федерации от 27.12....,28.12.2020,NaN,https://t.me/guberniaband; https://t.me/iwantm...,Физические лица,NaN,602723196108,NaN,...,22.02.1985,NaN,NaN,NaN,40817810451861520328,Псковское отделение №8630 ПАО Сбербанк,045805602,30101810300000000602,01.03.2025,NaN


In [5]:
test_cases = [
    'Иванов Иван Иванович',
    'Иванов Иван Иванович "ivanov"',
    'Иванов Иван "ivan"',
    'Петров Петr',
    '"superuser" Сидоров Сидор Сидорович',
    '"only_nickname"',
    'Смирнова Анна "анютка"',
    'Кузнецов "kuznec"'
]
international_tests = [
    'Smith John William',
    'Garcia Maria "maria_g"',
    'van der Berg Pieter',
    'O\'Connor Michael',
    'Li Wei "lw123"'
]

def universal_extract_last_name(full_name):
    """
    Универсальная функция для извлечения фамилии
    """
    if not full_name or not isinstance(full_name, str):
        return None
    
    # Удаляем никнеймы в кавычках
    cleaned_name = re.sub(r'"[^"]*"', '', full_name)
    
    # Удаляем лишние пробелы
    cleaned_name = re.sub(r'\s+', ' ', cleaned_name).strip()
    
    if not cleaned_name:
        return None
    
    # Разбиваем на слова
    words = cleaned_name.split()
    
    if not words:
        return None
    
    # Возвращаем первое слово
    return words[0]

# Тестирование всех случаев
all_tests = test_cases + international_tests

print("\nУниверсальная версия:")
for test in all_tests:
    result = universal_extract_last_name(test)
    print(f"'{test}' -> '{result}'")


Универсальная версия:
'Иванов Иван Иванович' -> 'Иванов'
'Иванов Иван Иванович "ivanov"' -> 'Иванов'
'Иванов Иван "ivan"' -> 'Иванов'
'Петров Петr' -> 'Петров'
'"superuser" Сидоров Сидор Сидорович' -> 'Сидоров'
'"only_nickname"' -> 'None'
'Смирнова Анна "анютка"' -> 'Смирнова'
'Кузнецов "kuznec"' -> 'Кузнецов'
'Smith John William' -> 'Smith'
'Garcia Maria "maria_g"' -> 'Garcia'
'van der Berg Pieter' -> 'van'
'O'Connor Michael' -> 'O'Connor'
'Li Wei "lw123"' -> 'Li'


In [6]:
def create_last_name_variations(last_name):
    """
    Создает возможные варианты склонения фамилии
    """
    morph = pymorphy3.MorphAnalyzer()
    variations = set()
    
    # Парсим фамилию
    parsed = morph.parse(last_name)[0]
    
    # Добавляем нормальную форму
    variations.add(parsed.normal_form.lower())
    
    # Генерируем варианты в разных падежах
    cases = ['nomn', 'gent', 'datv', 'accs', 'ablt', 'loct']
    
    for case in cases:
        try:
            inflected = parsed.inflect({case})
            if inflected:
                variations.add(inflected.word.lower())
        except:
            continue
    
    return variations

def find_last_name_in_text(text, target_last_names):
    """
    Ищет фамилию в тексте и возвращает найденную фамилию или "Not Found"
    """
    if not text or not target_last_names:
        return "Not Found"
    
    # Создаем множество всех возможных вариантов фамилий
    all_variations = {}
    for last_name in target_last_names:
        variations = create_last_name_variations(last_name)
        for variation in variations:
            all_variations[variation] = last_name  # Сохраняем оригинальную фамилию
    
    # Приводим текст к нижнему регистру и ищем совпадения
    text_lower = text.lower()
    words = re.findall(r'\b[а-яё]+\b', text_lower)
    
    for word in words:
        if word in all_variations:
            return all_variations[word]  # Возвращаем оригинальную фамилию
    
    return "Not Found"

# Пример использования
text = "Вчера я видел Иванова, который разговаривал с Ивановой о делах Петрова"
targets = ["Иванов", "Петров"]

result = find_last_name_in_text(text, targets)
print(f"Найдены фамилии: {result}")

# Покажем сгенерированные варианты
print("\nВарианты фамилии 'Иванов':")
print(create_last_name_variations("Моргенштерн"))

Найдены фамилии: Иванов

Варианты фамилии 'Иванов':
{'моргенштерном', 'моргенштерн', 'моргенштерне', 'моргенштерну', 'моргенштерна'}


In [7]:
#Подготовим стоп-список имён 

stop_list = df['Unnamed: 1'].to_list()

print(stop_list)

cleaned_name = ''
initials_full_sl = []

#сначала нужно сделать список только из ФИО
for name in stop_list:
    # Удаляем никнеймы в кавычках
    cleaned_name = re.sub(r'"[^"]*"', '', name)
    
    # Удаляем лишние пробелы
    cleaned_name = re.sub(r'\s+', ' ', cleaned_name).strip()

    initials_full_sl.append(cleaned_name)


#Оставим только фамилии
stop_list_lastnames = []

for names in stop_list:
    stop_list_lastnames.append(universal_extract_last_name(names))

print(stop_list_lastnames)


['Степанова Анна Васильевна', 'Смолин Владимир Александрович', 'Ицхоки Олег Евгеньевич', 'Говорун Сергей Николаевич', 'Несмиян Анатолий Евгеньевич "El Murid"', 'Кузнецов Борис Аврамович', 'Кротенко Евгений Андреевич', 'Кривцова Олеся Романовна', 'Везикко Ирия Валтеровна', 'Бобров Юрий Сергеевич', 'Нюберг Дмитрий Сергеевич "Qianti"', 'Марков Сергей Александрович', 'Кордочкин Андрей Борисович', 'Дугарова Раджана Дашинимаевна', 'Солонин Марк Семенович', 'Рудников Игорь Петрович', 'Курмояров Иоанн Валерьевич', 'Кротов Марк Яковлевич "Крутов"', 'Сивенок Андрей Александрович', 'Мысина Оксана Анатольевна', 'Мовчан Андрей Андреевич', 'Долиев Михаил Вячеславович', 'Храмцов Дмитрий Александрович', 'Торстрем Ксения Владимировна', 'Резник Наталья Рэмовна', 'Ольшевец Фелицата Дмитриевна "Маша Майерс"', 'Кириллова Ксения Валерьевна', 'Жарков Василий Павлович', 'Великий Дмитрий Сергеевич', 'Росов Николай Вячеславович', 'Поташов Валерий Николаевич', 'Литвишко Дарья Алексеевна', 'Зубарева Раиса Дмитрие

In [8]:
print(initials_full_sl)

['Степанова Анна Васильевна', 'Смолин Владимир Александрович', 'Ицхоки Олег Евгеньевич', 'Говорун Сергей Николаевич', 'Несмиян Анатолий Евгеньевич', 'Кузнецов Борис Аврамович', 'Кротенко Евгений Андреевич', 'Кривцова Олеся Романовна', 'Везикко Ирия Валтеровна', 'Бобров Юрий Сергеевич', 'Нюберг Дмитрий Сергеевич', 'Марков Сергей Александрович', 'Кордочкин Андрей Борисович', 'Дугарова Раджана Дашинимаевна', 'Солонин Марк Семенович', 'Рудников Игорь Петрович', 'Курмояров Иоанн Валерьевич', 'Кротов Марк Яковлевич', 'Сивенок Андрей Александрович', 'Мысина Оксана Анатольевна', 'Мовчан Андрей Андреевич', 'Долиев Михаил Вячеславович', 'Храмцов Дмитрий Александрович', 'Торстрем Ксения Владимировна', 'Резник Наталья Рэмовна', 'Ольшевец Фелицата Дмитриевна', 'Кириллова Ксения Валерьевна', 'Жарков Василий Павлович', 'Великий Дмитрий Сергеевич', 'Росов Николай Вячеславович', 'Поташов Валерий Николаевич', 'Литвишко Дарья Алексеевна', 'Зубарева Раиса Дмитриевна', 'Елизаров Павел Олегович', 'Трикоз Ев

In [9]:
with open("inoagents_050925.txt", "w", encoding="utf-8") as file:
    file.writelines(', '.join(stop_list_lastnames))

with open("ia_initials_050925.txt", "w", encoding="utf-8") as file:
    file.writelines(', '.join(initials_full_sl))

print("Список сохранен в файл my_list.txt")

Список сохранен в файл my_list.txt
